# Gas Turbine Cogeneration System - Functional Approach

This notebook demonstrates how to run the refactored gas turbine cogeneration system using a functional programming approach (without Python classes).

The model is based on the IDAES framework and simulates the CGAM (Cogeneration problem from the Chemical Gas turbine Advanced Modeling) process.

## 1. Import Required Libraries

In [ ]:
import os
import logging
from pathlib import Path
import pandas as pd

import pyomo.environ as pyo

import idaes
import idaes.core.util.scaling as iscale
from idaes.core.solvers import use_idaes_solver_configuration_defaults

# Import the functional approach functions from script.py
from script import (
    build_flowsheet,
    initialize_flowsheet,
    steam_streams_dataframe,
    flue_gas_streams_dataframe,
    check_scaling,
    make_directory
)

# Set logging level
logging.getLogger("pyomo").setLevel(logging.ERROR)

## 2. Setup Directories for Data Output

In [ ]:
# Create directories for saving data
make_directory("data")
make_directory("data_pfds")
make_directory("data_tabulated")

print("Directories created successfully!")

## 3. Configure IPOPT Solver

In [ ]:
# Use IDAES default solver configuration
use_idaes_solver_configuration_defaults()

# Configure IPOPT solver options
idaes.cfg.ipopt.options.nlp_scaling_method = "user-scaling"
idaes.cfg.ipopt.options.linear_solver = "ma57"
idaes.cfg.ipopt.options.ma57_pivtol = 1e-5
idaes.cfg.ipopt.options.ma57_pivtolmax = 0.1

# Create solver
solver = pyo.SolverFactory("ipopt")

print("Solver configured successfully!")

## 4. Build the Flowsheet

This step creates the complete gas turbine flowsheet including:
- Property packages (air, combustion gases, flue gas, water/steam)
- Unit models (compressor, turbine, combustor, heat exchangers, etc.)
- Constraints (pressure drops, power balance, etc.)
- Arcs (connections between units)
- Initial inputs and boundary conditions
- Tags for stream tables
- Scaling factors

In [ ]:
# Create Pyomo concrete model
m = pyo.ConcreteModel()

# Build the flowsheet
fs = build_flowsheet(m)

print("Flowsheet built successfully!")
print(f"Number of variables: {m.nvariables()}")
print(f"Number of constraints: {m.nconstraints()}")

## 5. Calculate Scaling Factors

In [ ]:
# Calculate scaling factors for better numerical performance
iscale.calculate_scaling_factors(m)

print("Scaling factors calculated!")

## 6. Initialize the Flowsheet

This step initializes all unit models in the flowsheet using sequential decomposition.

In [ ]:
# Initialize the flowsheet
# The initialization will save the state to a file that can be reloaded later
initialize_flowsheet(
    fs,
    load_from="gas_turbine_init.json.gz",  # Try to load from this file if it exists
    save_to="gas_turbine_init.json.gz",     # Save initialized state to this file
)

print("Flowsheet initialized successfully!")

## 7. Solve the Model

In [ ]:
# Solve the optimization problem
res = solver.solve(m, tee=True)

print("\n" + "="*60)
print("Solver Status:", res.solver.status)
print("Termination Condition:", res.solver.termination_condition)
print("="*60)

## 8. Display Results - Steam Streams

In [ ]:
# Generate and display steam stream table
steam_df = steam_streams_dataframe(fs)
print("\n=== STEAM STREAMS ===")
display(steam_df)

## 9. Display Results - Flue Gas Streams

In [ ]:
# Generate and display flue gas stream table
flue_gas_df = flue_gas_streams_dataframe(fs)
print("\n=== FLUE GAS STREAMS ===")
display(flue_gas_df)

## 10. Display Key Performance Metrics

In [ ]:
# Extract and display key performance metrics
print("\n=== KEY PERFORMANCE METRICS ===")
print(f"Compressor Power: {pyo.value(fs.cmp1.control_volume.work[0])/1e6:.2f} MW")
print(f"Gas Turbine Stage 1 Power: {pyo.value(fs.gts1.control_volume.work[0])/1e6:.2f} MW")
print(f"Total GT Power Output: {-pyo.value(fs.gt_power[0])/1e6:.2f} MW")
print(f"Compressor Isentropic Efficiency: {pyo.value(fs.cmp1.efficiency_isentropic[0])*100:.2f}%")
print(f"Turbine Isentropic Efficiency: {pyo.value(fs.gts1.efficiency_isentropic[0])*100:.2f}%")
print(f"Combustor Temperature: {pyo.value(fs.cmb1.control_volume.properties_out[0].temperature):.2f} K")
print(f"Fuel Mass Flow: {pyo.value(fs.feed_fuel1.properties[0].flow_mass):.3f} kg/s")

## 11. Check Model Scaling (Optional)

This cell checks the quality of the model scaling, which is important for numerical stability.

In [ ]:
# Check scaling quality
print("\n=== SCALING DIAGNOSTICS ===")
check_scaling(fs, m)

## 12. Save Results to CSV (Optional)

In [ ]:
# Save stream tables to CSV files
steam_df.to_csv("data_tabulated/steam_streams.csv")
flue_gas_df.to_csv("data_tabulated/flue_gas_streams.csv")

print("Results saved to CSV files!")

## Summary

This notebook demonstrates the complete workflow for:
1. Building a gas turbine cogeneration system model using a functional approach
2. Initializing and solving the model
3. Displaying and analyzing the results

All the functionality is provided by standalone functions from `script.py`, eliminating the need for Python classes.

### Key Functions Used:
- `build_flowsheet(m)`: Builds the complete flowsheet
- `initialize_flowsheet(fs, ...)`: Initializes all unit models
- `steam_streams_dataframe(fs)`: Generates steam stream table
- `flue_gas_streams_dataframe(fs)`: Generates flue gas stream table
- `check_scaling(fs, m)`: Checks model scaling quality